# 1. Base Setup

## 1.1 Install packages

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost
!pip install lightgbm

## 1.2 Load all needed imports

In [12]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix, accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [13]:
import cf_copilot
print(cf_copilot.__file__)

/home/jeroenewalts/code/EwaltsJ/cf_copilot/cf_copilot/__init__.py


In [14]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [15]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)

# Example: train+valid for search, final holdout for test
valid_cutoff = big_df["reference_date"].quantile(0.6)
test_cutoff = big_df["reference_date"].quantile(0.8)

search_df = big_df[big_df["reference_date"] <= test_cutoff].copy()
final_test_df = big_df[big_df["reference_date"] > test_cutoff].copy()

test_fold = np.where(search_df["reference_date"] <= valid_cutoff, -1, 0)
ps = PredefinedSplit(test_fold)

X_search, y_search = preprocess(search_df)
X_final_test, y_final_test = preprocess(final_test_df)

Loading local file: /home/jeroenewalts/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [16]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [ ]:
from lightgbm import LGBMClassifier

classifier = LGBMClassifier(
    objective="multiclass",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", classifier),
])

In [18]:
pipeline.get_params()

{'memory': None,
 'steps': [('preprocessor',
   ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                    ['business_year', 'invoice_age_days',
                                     'days_until_due', 'pay_terms_days',
                                     'invoice_month', 'due_month', 'days_past_due',
                                     'customer_avg_delay', 'late_payment_ratio',
                                     'prev_transaction_count',
                                     'days_since_last_invoice',
                                     'customer_risk_score', 'invoice_amount',
                                     'invoice_amount_log']),
                                   ('cat',
                                    Pipeline(steps=[('imputer',
                                                     SimpleImputer(fill_value=-1,
                                                                   strategy='constant')),
                     

In [ ]:
import datetime
print("START:", datetime.datetime.now())

param_distributions = {
    # Core boosting capacity
    "classifier__n_estimators": randint(300, 1801),
    "classifier__learning_rate": uniform(0.01, 0.14),
    "classifier__num_leaves": randint(15, 128),
    "classifier__max_depth": [-1, 4, 6, 8, 10, 12],

    # Split / leaf control
    "classifier__min_child_samples": randint(10, 101),
    "classifier__min_child_weight": uniform(1e-3, 0.2),
    "classifier__min_split_gain": uniform(0.0, 0.2),

    # Sampling
    "classifier__subsample": uniform(0.7, 0.3),
    "classifier__subsample_freq": randint(1, 8),
    "classifier__colsample_bytree": uniform(0.7, 0.3),

    # Regularization
    "classifier__reg_alpha": uniform(0.0, 2.0),
    "classifier__reg_lambda": uniform(0.0, 2.0),
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=60,
    cv=ps,
    scoring="neg_log_loss",
    n_jobs=6,
    verbose=2,
    random_state=42,
    error_score=np.nan,
    return_train_score=True,
)

random_search.fit(X_search, y_search)

best_model = random_search.best_estimator_

In [ ]:
y_proba = best_model.predict_proba(X_final_test)
y_pred = best_model.predict(X_final_test)

clf_classes = np.array(best_model.named_steps["classifier"].classes_)

search_classes = np.array(sorted(pd.Series(y_search).unique()))
test_classes = np.array(sorted(pd.Series(y_final_test).unique()))

cv_results = pd.DataFrame(random_search.cv_results_)
n_failed = cv_results["mean_test_score"].isna().sum()
print(f"Failed parameter sets: {n_failed} / {len(cv_results)}")

missing_in_train = set(test_classes) - set(clf_classes)
print("Classes in final test but not seen during search:", missing_in_train)

if missing_in_train:
    raise ValueError(
        f"Final test contains unseen classes: {missing_in_train}. "
        "The model cannot evaluate those correctly."
    )

if y_proba.shape[1] != len(clf_classes):
    raise ValueError(
        f"Mismatch between probability columns ({y_proba.shape[1]}) "
        f"and classifier classes ({len(clf_classes)})."
    )

print("Best CV log_loss:", -random_search.best_score_)
print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

final_log_loss = log_loss(y_final_test, y_proba, labels=clf_classes)
final_accuracy = accuracy_score(y_final_test, y_pred)
final_f1_macro = f1_score(y_final_test, y_pred, average="macro")

print("\nFinal holdout performance")
print(f"log_loss  : {final_log_loss:.6f}")
print(f"accuracy  : {final_accuracy:.6f}")
print(f"f1_macro  : {final_f1_macro:.6f}")

print("\nClass checks")
print("Search classes:", list(search_classes))
print("Test classes  :", list(test_classes))
print("Model classes :", list(clf_classes))

best_idx = random_search.best_index_
best_train_score = random_search.cv_results_["mean_train_score"][best_idx]
best_valid_score = random_search.cv_results_["mean_test_score"][best_idx]

print("\nBest CV split scores")
print(f"Best mean train neg_log_loss: {best_train_score:.6f}")
print(f"Best mean valid neg_log_loss: {best_valid_score:.6f}")
print(f"Best mean train log_loss    : {-best_train_score:.6f}")
print(f"Best mean valid log_loss    : {-best_valid_score:.6f}")

display(
    cv_results.sort_values("rank_test_score")[
        ["rank_test_score", "mean_test_score", "mean_train_score", "params"]
    ].head(5)
)

In [ ]:
from cf_copilot.cashflow_prediction.evaluation import evaluate_forecast_holdout

# Small compatibility fix:
# evaluate_forecast_holdout expects model.classes_,
# but best_model is a Pipeline and stores classes on the classifier step
best_model.classes_ = best_model.named_steps["classifier"].classes_

forecast_metrics, forecast_summary = evaluate_forecast_holdout(
    model=best_model,
    test_df=final_test_df,
    verbose=True,
)

print("\nAggregate forecast metrics")
for k, v in forecast_metrics.items():
    print(f"{k}: {v}")

pd.DataFrame([forecast_metrics])

print("END:", datetime.datetime.now())